In [1]:
!pip -q install pypdf sentence-transformers faiss-cpu

**Explanation: Installing Required Libraries**

This code installs three Python libraries required for building an AI application.

- **pypdf:** Extracts text from PDF documents.
- **sentence-transformers:** Converts text into numerical embeddings for semantic similarity.
- **faiss-cpu:** Enables efficient similarity search to retrieve relevant text.

The `!pip install` command installs the libraries in Google Colab, while `-q` reduces installation output.

These libraries are useful for building a Retrieval-Augmented Generation (RAG) system that retrieves relevant information from documents to answer user queries.

In [2]:
policy = """
COMPANY HR POLICY

Annual Leave:
Employees are entitled to 24 days of annual leave per calendar year.
Leave requests should normally be submitted at least 5 working days
in advance.

Sick Leave:
Employees who are unable to work due to illness should inform their
manager before the start of their working day. A medical certificate
may be required for extended periods of absence.

Paternity Leave:
Eligible employees are entitled to 15 working days of paternity leave.
The leave should be taken within six months of the child's birth.

Work From Home:
Employees may work from home up to 2 days per week with manager approval.
Employees must remain available during normal working hours.

Travel Expenses:
Employees travelling for business can claim reasonable hotel,
transportation, and meal expenses. Original receipts should be
submitted within 30 days of returning from the trip.
"""

In [3]:
chunks = [
    paragraph.strip()
    for paragraph in policy.split("\n")
    if paragraph.strip()
]
for i, chunk in enumerate(chunks):
  print(f"{i}:{chunk}")
  print(chunk)
  print("-"*40)

0:COMPANY HR POLICY
COMPANY HR POLICY
----------------------------------------
1:Annual Leave:
Annual Leave:
----------------------------------------
2:Employees are entitled to 24 days of annual leave per calendar year.
Employees are entitled to 24 days of annual leave per calendar year.
----------------------------------------
3:Leave requests should normally be submitted at least 5 working days
Leave requests should normally be submitted at least 5 working days
----------------------------------------
4:in advance.
in advance.
----------------------------------------
5:Sick Leave:
Sick Leave:
----------------------------------------
6:Employees who are unable to work due to illness should inform their
Employees who are unable to work due to illness should inform their
----------------------------------------
7:manager before the start of their working day. A medical certificate
manager before the start of their working day. A medical certificate
-------------------------------------

**Explanation: Splitting Text into Chunks**

This code splits the policy text into smaller chunks using newline characters (`\n`). The `strip()` function removes extra spaces from the beginning and end of each paragraph, and empty paragraphs are ignored.

The `for` loop prints each chunk with its index number, followed by the paragraph text and a separator line of 40 hyphens.

In [5]:
# Generate Embeddings

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedding_model.encode(chunks)

print("Number of Chunks:", len(chunks))
print("shape of embeddings:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of Chunks: 19
shape of embeddings: (19, 384)


In [6]:
#Create the Vector Database

import faiss
import numpy as np

# Ensure chunks is defined
if 'chunks' not in globals():
    policy = """
COMPANY HR POLICY

Annual Leave:
Employees are entitled to 24 days of annual leave per calendar year.
Leave requests should normally be submitted at least 5 working days
in advance.

Sick Leave:
Employees who are unable to work due to illness should inform their
manager before the start of their working day. A medical certificate
may be required for extended periods of absence.

Paternity Leave:
Eligible employees are entitled to 15 working days of paternity leave.
The leave should be taken within six months of the child's birth.

Work From Home:
Employees may work from home up to 2 days per week with manager approval.
Employees must remain available during normal working hours.

Travel Expenses:
Employees travelling for business can claim reasonable hotel,
transportation, and meal expenses. Original receipts should be
submitted within 30 days of returning from the trip.
"""
    chunks = [
        paragraph.strip()
        for paragraph in policy.split("\n")
        if paragraph.strip()
    ]

# Ensure embeddings are defined
if 'embeddings' not in globals():
    if 'embedding_model' not in globals():
        from sentence_transformers import SentenceTransformer
        embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = embedding_model.encode(chunks)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

In [7]:
question = "How many days of paternity leave can I take?"
print("Vectors Stored:",index.ntotal)

Vectors Stored: 19


In [8]:
question_embedding = embedding_model.encode(
    [question]
)

In [9]:
distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    2
)

In [10]:
for i in indices[0]:
  print(chunks[i])
  print("-" * 50)

Eligible employees are entitled to 15 working days of paternity leave.
--------------------------------------------------
Paternity Leave:
--------------------------------------------------


In [11]:
def retreive_context(question, top_k=2):
  question_embedding = embedding_model.encode(
      [question]
  )

  distances, indices = index.search(
      np.array(question_embedding).astype("float32"),
      top_k

  )

  results = []

  for i in indices[0]:
    results.append(chunks[i])


  return results

In [12]:
question = "Can emmployees work from home?"

results = retreive_context(question)

for result in results:
  print(result)

Work From Home:
Employees may work from home up to 2 days per week with manager approval.


In [13]:
question = "can emmpolyees  take 20 days of maternity leave?"

results = retreive_context(question)

for result in results:
  print(result)

The leave should be taken within six months of the child's birth.
Eligible employees are entitled to 15 working days of paternity leave.


In [14]:
question = "Can emmployees quite the job?"

results = retreive_context(question)

for result in results:
  print(result)

Work From Home:
Sick Leave:


In [15]:
!pip -q install gradio

In [16]:

import gradio as gr
import numpy as np

def search_policy(question, top_k):
    if not question or not question.strip():
        return "⚠️ Please enter a question first."

    try:
        results = retreive_context(
            question.strip(),
            top_k=int(top_k)
        )

        if not results:
            return "🔍 No relevant policy information found."

        output = "## 📚 Relevant Policy Information\n\n"

        for i, result in enumerate(results, start=1):
            output += (
                f"### 📄 Result {i}\n\n"
                f"> {result}\n\n"
                f"---\n\n"
            )

        return output

    except Exception as e:
        return f"❌ **Error:** `{type(e).__name__}: {e}`"


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="indigo",
        neutral_hue="slate"
    ),
    title="HR Policy Assistant"
) as demo:

    gr.Markdown(
        """
        # 🏢 HR Policy Assistant
        ### 💬 Your smart companion for company policies

        Find relevant information about leave, work from home,
        employee benefits, and travel policies.
        """
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=180):
            gr.Markdown("## ⚙️ Settings")

            top_k = gr.Slider(
                minimum=1,
                maximum=5,
                value=2,
                step=1,
                label="🔎 Results to retrieve"
            )

            gr.Markdown(
                """
                ### 💡 Tips
                - Ask clear questions.
                - Use specific policy terms.
                - Check the retrieved policy text.
                """
            )

        with gr.Column(scale=3):
            question = gr.Textbox(
                label="✍️ Ask your question",
                placeholder="e.g. How many days of annual leave are allowed?",
                lines=3
            )

            with gr.Row():
                search_btn = gr.Button(
                    "🔍 Search Policy",
                    variant="primary"
                )
                clear_btn = gr.ClearButton(
                    components=[question],
                    value="🗑️ Clear"
                )

            gr.Markdown("### ✨ Try these questions")

            gr.Examples(
                examples=[
                    ["How many days of annual leave are allowed?"],
                    ["Can employees work from home?"],
                    ["How much paternity leave is provided?"],
                    ["When should travel receipts be submitted?"],
                    ["What should an employee do when sick?"]
                ],
                inputs=question
            )

    gr.Markdown("---")
    output = gr.Markdown(
        "### 📋 Results will appear here\nEnter a question above to get started."
    )

    search_btn.click(
        fn=search_policy,
        inputs=[question, top_k],
        outputs=output
    )

    question.submit(
        fn=search_policy,
        inputs=[question, top_k],
        outputs=output
    )

demo.launch(share=True)


/tmp/ipykernel_1106/3134509968.py:32: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f578636cec29e62dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
def retreive_context(question, top_k=2):
    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        top_k
    )

    results = []

    for i in indices[0]:
        if i >= 0:
            results.append(chunks[i])

    return results

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Ensure the model and numpy are initialized
if 'embedding_model' not in globals():
    embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
if 'np' not in globals():
    import numpy as np

print(retreive_context("Can employees work from home?"))

['Work From Home:', 'Employees may work from home up to 2 days per week with manager approval.']
